# NBA Game Data — EDA

Read raw game data from S3 and explore.

In [1]:
import json
import boto3
import pandas as pd

S3_BUCKET = "prediction-markets-data"
s3 = boto3.client("s3")

In [2]:
# List what's in the bucket
resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix="nba/")
for obj in resp.get("Contents", []):
    print(f"{obj['Key']}  ({obj['Size'] / 1024:.0f} KB)")

nba/games/season_2024-25.json  (1219 KB)


In [3]:
# Load a season from S3
SEASON = "2024-25"

obj = s3.get_object(Bucket=S3_BUCKET, Key=f"nba/games/season_{SEASON}.json")
raw = json.loads(obj["Body"].read())
df = pd.DataFrame(raw)
print(f"{len(df)} rows, {df['GAME_ID'].nunique()} games")
df.head()

2802 rows, 1401 games


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
0,42024,1610612760,OKC,Oklahoma City Thunder,0042400407,2025-06-22,OKC vs. IND,W,240,103,...,0.710,13,27,40,20,14,8,7,23,12.0
1,42024,1610612754,IND,Indiana Pacers,0042400407,2025-06-22,IND @ OKC,L,240,91,...,0.759,12,33,45,17,6,4,21,24,-12.0
2,42024,1610612760,OKC,Oklahoma City Thunder,0042400406,2025-06-19,OKC @ IND,L,240,91,...,0.808,4,37,41,14,4,4,21,20,-17.0
3,42024,1610612754,IND,Indiana Pacers,0042400406,2025-06-19,IND vs. OKC,W,240,108,...,0.680,11,35,46,23,16,5,10,17,17.0
4,42024,1610612760,OKC,Oklahoma City Thunder,0042400405,2025-06-16,OKC vs. IND,W,239,120,...,0.813,19,26,45,24,15,12,11,24,11.0


In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
# Win/loss distribution by team
record = df.groupby("TEAM_ABBREVIATION")["WL"].value_counts().unstack(fill_value=0)
record["win_pct"] = record["W"] / (record["W"] + record["L"])
record.sort_values("win_pct", ascending=False)

In [ ]:
# Points per game by team
ppg = df.groupby("TEAM_ABBREVIATION")["PTS"].mean().sort_values(ascending=False)
ppg.plot.bar(title="Points Per Game", figsize=(14, 4))

In [ ]:
# Scoring distribution
df["PTS"].hist(bins=30, figsize=(10, 4))

In [ ]:
# Home vs away win rates
df["is_home"] = df["MATCHUP"].str.contains("vs.")
home_wr = df.groupby("is_home")["WL"].value_counts(normalize=True).unstack()
home_wr